<a href="https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:


import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice

Method choice: I chose Logistic Regression because this task involves predicting a binary outcome and the model provides a simple and interpretable baseline for comparison. It allows the effect of the available features to be examined without introducing unnecessary model complexity.

The model will produce a probability of decline for each record. I will rank records by this probability and evaluate the top 20 using Precision@20, matching the Week-4 baseline metric. This makes the comparison focused on the same practical question: whether the highest-priority records are actually declining.

I will not use trend_direction, trend_pct, or variables representing future/previous outcome windows because they could leak information about the target.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
)

RANDOM_STATE = 42


df = pd.read_csv(("data/raw/content_refresh_anonymized.csv")
)

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

# Audit target only
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

Dataset shape: (30000, 44)
Columns: 44

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: I used the same train/test data and evaluation metric as the Week-4 baseline so that the comparison is fair. The baseline was recreated using the training data to calculate the position-tier benchmark and then evaluated on the test set. The Logistic Regression model was evaluated on the same test set using Precision@20.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# Columns that must not be used as predictive features
LEAKAGE_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

X = df.drop(columns=LEAKAGE_COLUMNS)
y = df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain decline rate:", y_train.mean())
print("Test decline rate:", y_test.mean())

Train shape: (24000, 42)
Test shape: (6000, 42)

Train decline rate: 0.5420833333333334
Test decline rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Result: Logistic Regression achieved a Precision@20 of 1.000 compared with 0.650 for the Week-4 rule baseline. This is an observed improvement of 0.350 Precision@20 points. Therefore, on this test split and metric, Logistic Regression performed better than the Week-4 baseline.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# Columns that must not be used as predictive features
LEAKAGE_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

X = df.drop(columns=LEAKAGE_COLUMNS)
y = df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain decline rate:", y_train.mean())
print("Test decline rate:", y_test.mean())


# Identify feature types
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number", "bool"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

print("Model training complete.")

Train shape: (24000, 42)
Test shape: (6000, 42)

Train decline rate: 0.5420833333333334
Test decline rate: 0.542
Numeric features: 29
Categorical features: 13

Categorical columns:
['content_id', 'client_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']
Model training complete.


In [14]:
#comparing against the week-4 baseline
def precision_at_k(y_true, scores, k=20):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()




In [15]:
model_scores = model.predict_proba(X_test)[:, 1]

model_p20 = precision_at_k(
    y_test,
    model_scores,
    k=20
)

print("Model Precision@20:", round(model_p20, 3))

Model Precision@20: 1.0


In [16]:
#recreating the week-4 baseline
train_reference = X_train.copy()
train_reference["target"] = y_train.values

tier_benchmark = (
    train_reference
    .groupby("position_tier")["ctr"]
    .median()
    .to_dict()
)

tier_benchmark


baseline_test = X_test.copy()

baseline_test["tier_benchmark_ctr"] = (
    baseline_test["position_tier"]
    .map(tier_benchmark)
)

baseline_test["baseline_flag"] = (
    (baseline_test["position_tier"] != "no_data") &
    (baseline_test["impressions_90d"] >= 500) &
    (
        baseline_test["ctr"] <
        0.5 * baseline_test["tier_benchmark_ctr"]
    )
)

baseline_test["baseline_score"] = (
    baseline_test["baseline_flag"].astype(int)
    * baseline_test["impressions_90d"]
)

baseline_p20 = precision_at_k(
    y_test,
    baseline_test["baseline_score"],
    k=20
)

print("Baseline Precision@20:", round(baseline_p20, 3))


Baseline Precision@20: 0.65


In [17]:
#model -vs- baseline table
results = pd.DataFrame({
    "Method": [
        "Week-4 rule baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ]
})

results["Improvement_vs_baseline"] = (
    results["Precision@20"] - baseline_p20
)

results

results.style.format({
    "Precision@20": "{:.3f}",
    "Improvement_vs_baseline": "{:+.3f}"
})

,Method,Precision@20,Improvement_vs_baseline
0,Week-4 rule baseline,0.650,+0.000
1,Logistic Regression,1.000,+0.350


In [18]:
model_predictions = (model_scores >= 0.5).astype(int)

print("ROC-AUC:", round(
    roc_auc_score(y_test, model_scores), 3
))

print("Precision:", round(
    precision_score(y_test, model_predictions, zero_division=0), 3
))

print("Recall:", round(
    recall_score(y_test, model_predictions, zero_division=0), 3
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, model_predictions))

ROC-AUC: 0.922
Precision: 0.847
Recall: 0.846

Confusion matrix:
[[2252  496]
 [ 502 2750]]


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error analysis: Logistic Regression achieved 1.000 Precision@20, meaning all 20 records selected at the top of the ranking were relevant according to the test target. Therefore, there were no false-positive errors within the top-20 set. However, this result only evaluates the highest-ranked 20 records and should not be interpreted as perfect performance across the entire test set. The position and CTR signal analysis also showed that mean CTR decreased as position became worse, while freshness and impressions showed less consistent relationships with decline rate. These findings provide useful directional evidence for interpreting the model rather than assuming that every feature has the same predictive value.

The Logistic Regression coefficients indicate that recent impression-related features were among the strongest model signals. impressions_prev_30d had the largest positive coefficient, while impressions_last_30d had the largest negative coefficient. These coefficients describe associations used by the model and should not be interpreted as causal effects.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted_probability"] = model_scores
error_analysis["predicted_class"] = model_predictions

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) &
        (error_analysis["predicted_class"] == 0),

        (error_analysis["actual"] == 0) &
        (error_analysis["predicted_class"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

error_analysis["error_type"].value_counts()

,count
error_type,
correct,5002
false_negative,502
false_positive,496


In [20]:
top_20 = (
    error_analysis
    .sort_values("predicted_probability", ascending=False)
    .head(20)
)

top_20[[
    "predicted_probability",
    "actual",
    "error_type"
]]

,predicted_probability,actual,error_type
1448,1.0,1,correct
12623,1.0,1,correct
29660,1.0,1,correct
25054,1.0,1,correct
18509,1.0,1,correct
17235,1.0,1,correct
17912,1.0,1,correct
6653,1.0,1,correct
21261,1.0,1,correct
21819,1.0,1,correct


Feature interpretation: The model relied strongly on recent impression and click features. Some client and content ID categories also had relatively large coefficients, indicating that entity-specific patterns contributed to predictions. These coefficients should be interpreted as model associations rather than causal effects.

In [21]:
#feature interpretation
feature_names = (
    model.named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    model.named_steps["classifier"]
    .coef_[0]
)

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values(
    "abs_coefficient",
    ascending=False
)

importance.head(20)

print("Top positive features:")
print(
    importance
    .sort_values("coefficient", ascending=False)
    .head(10)[["feature", "coefficient"]]
)

print("\nTop negative features:")
print(
    importance
    .sort_values("coefficient")
    .head(10)[["feature", "coefficient"]]
)

Top positive features:
                                            feature  coefficient
18                    numeric__impressions_prev_30d    25.917816
5                          numeric__impressions_90d     1.827957
24043      categorical__client_id_client_7f2253d7e2     1.417789
24047      categorical__client_id_client_9400f1b21c     1.350637
24036      categorical__client_id_client_3fdba35f04     1.348375
19                         numeric__clicks_prev_30d     1.321468
1885   categorical__content_id_content_13bbd72aea33     0.844875
23062  categorical__content_id_content_f616ca0ec5ea     0.832014
13659  categorical__content_id_content_933296f93aa5     0.817611
20664  categorical__content_id_content_dd04e6df7ef9     0.814093

Top negative features:
                                            feature  coefficient
15                    numeric__impressions_last_30d   -32.047070
24033      categorical__client_id_client_25fc0e7096    -3.412665
24057      categorical__client_id_client_e2

In [22]:
#error summary

false_positive_rate = (
    (error_analysis["error_type"] == "false_positive").mean()
)

false_negative_rate = (
    (error_analysis["error_type"] == "false_negative").mean()
)

print(
    f"False-positive rate among all test records: "
    f"{false_positive_rate:.3f}"
)

print(
    f"False-negative rate among all test records: "
    f"{false_negative_rate:.3f}"
)

top20_errors = (
    top_20["error_type"]
    .value_counts()
)

print("Errors among top 20 ranked records:")
print(top20_errors)

False-positive rate among all test records: 0.083
False-negative rate among all test records: 0.084
Errors among top 20 ranked records:
error_type
correct    20
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.